# Sentiment classification: BPE, log counts and MLP

Compare text classification systems:

- **3 representations:** token indicators, bag of words, and **log counts**
- **3 architectures:** log-linear, single-hidden-layer MLP, and **small MLP** (one hidden layer of 16 units)
- **Evaluation:** cross-validation with a relevant metric

This notebook trains all models with cross-validation and saves the results to `../results/`.

## Setup

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import sys
import json
import warnings
from pathlib import Path
from collections import Counter, OrderedDict

import numpy as np
import torch

warnings.simplefilter(action='ignore', category=FutureWarning)

sys.path.insert(0, str(Path("../src").resolve()))  # experiment.py, clslib.py, loglin.py, smlp.py
import experiment
from clslib import Encoder, NamedClasses

results_folder = Path("../results")
models_folder = results_folder / "models"  # trained models and tokenizer
models_folder.mkdir(parents=True, exist_ok=True)

## 1. Data

We use **IMDB** ([mteb/imdb](https://huggingface.co/datasets/mteb/imdb)): movie reviews, labelled `negative` (0) or `positive` (1) (binary sentiment classification).

| split | reviews | labels |
|---|---|---|
| train | ~25,000 | yes |
| test | ~25,000 | yes |

We only use **train**, for cross-validation. The project does not ask for a test set evaluation, so **test** is not used.

Load the dataset from Hugging Face:

In [18]:
from datasets import load_dataset

DATASET_NAME = "mteb/imdb"

dataset = load_dataset(DATASET_NAME)

The two classes, with their numbers:

In [19]:
outcome_space = NamedClasses(['negative', 'positive'])
print(outcome_space)

0: negative, 1: positive


Split the data into texts and labels, for train and test:

In [20]:
TEXT_FIELD = "text"
LABEL_FIELD = "label"

train_texts = dataset["train"][TEXT_FIELD]
train_labels = dataset["train"][LABEL_FIELD]

test_texts = dataset["test"][TEXT_FIELD]
test_labels = dataset["test"][LABEL_FIELD]

Size of the data:

In [21]:
print("training examples:", len(train_texts))
print("test examples:", len(test_texts))

training examples: 24904
test examples: 24678


How many training examples each class has:

In [22]:
label_counts = Counter(train_labels)
for k, count in sorted(label_counts.items()):
    print(f"{outcome_space.name(k)}: {count} ({count / len(train_labels):.1%})")

negative: 12432 (49.9%)
positive: 12472 (50.1%)


## 2. Tokenizer

We train a BPE tokenizer on the training texts with a vocabulary of **5,000** tokens (the tutorial uses 1,000). With 1,000, many common words are cut into meaningless pieces (e.g. `st ud y`, `p ot at o es`); with 5,000, most common words stay whole.

In [ ]:
vocab_size = 5000
min_frequency = 2
tokenizer = experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
tokenizer.save(f"{models_folder}/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

Let's see how the tokenizer splits the first training review into tokens:

In [24]:
print(tokenizer.encode(train_texts[0]).tokens)

['▁I', '▁rented', '▁I', '▁A', 'M', '▁C', 'U', 'R', 'I', 'OU', 'S', '-', 'Y', 'E', 'LL', 'OW', '▁from', '▁my', '▁video', '▁store', '▁because', '▁of', '▁all', '▁the', '▁contro', 'vers', 'y', '▁that', '▁surround', 'ed', '▁it', '▁when', '▁it', '▁was', '▁first', '▁released', '▁in', '▁196', '7', '.', '▁I', '▁also', '▁heard', '▁that', '▁at', '▁first', '▁it', '▁was', '▁se', 'ized', '▁by', '▁U', '.S.', '▁c', 'ust', 'om', 's', '▁if', '▁it', '▁ever', '▁tried', '▁to', '▁ent', 'er', '▁this', '▁count', 'ry,', '▁there', 'fore', '▁being', '▁a', '▁fan', '▁of', '▁films', '▁considered', '▁"c', 'ont', 'ro', 'vers', 'ial', '"', '▁I', '▁really', '▁had', '▁to', '▁see', '▁this', '▁for', '▁myself', '.<br', '▁/><br', '▁/>The', '▁plot', '▁is', '▁cent', 'ered', '▁around', '▁a', '▁young', '▁Sw', 'ed', 'ish', '▁drama', '▁student', '▁named', '▁L', 'ena', '▁who', '▁wants', '▁to', '▁learn', '▁everything', '▁she', '▁can', '▁about', '▁life.', '▁In', '▁particular', '▁she', '▁wants', '▁to', '▁focus', '▁her', '▁att', 'ent'

## 3. Representations

All three have one dimension per token in the vocabulary:

- **Token indicator:** 1 if the token appears, 0 if not.
- **Bag of words:** how many times the token appears.
- **Log counts**: $\log(1 + \text{count})$, so very frequent tokens don't dominate.

In [25]:
class TokenIndicator(Encoder):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        return self.tokenizer.get_vocab_size()
    
    def __call__(self, x: str) -> 'np.array':
        vector = np.zeros(len(self))
        vector[list(set(self.tokenizer.encode(x).ids))] = 1
        return vector

class BagOfWords(Encoder):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        return self.tokenizer.get_vocab_size()
    
    def __call__(self, x: str) -> 'np.array':
        vector = np.zeros(len(self))
        np.add.at(vector, self.tokenizer.encode(x).ids, 1)
        return vector

class LogCounts(BagOfWords):
    def __call__(self, x: str) -> 'np.array':
        return np.log1p(super().__call__(x))

encoders = OrderedDict([
    ('indicator', TokenIndicator(tokenizer)),
    ('bow', BagOfWords(tokenizer)),
    ('logcounts', LogCounts(tokenizer)),
])

Let's encode the first training review with each encoder. `dim` is the vector size, `non-zero` is how many different tokens appear, and `sum` is the total of all values:

In [26]:
for name, encoder in encoders.items():
    h = encoder(train_texts[0])
    print(f"{name:>10}: dim={len(h)}, non-zero={int((h > 0).sum())}, sum={h.sum():.2f}")

 indicator: dim=5000, non-zero=253, sum=253.00
       bow: dim=5000, non-zero=253, sum=413.00
 logcounts: dim=5000, non-zero=253, sum=224.98


## 4. Cross-validation splits

We split the training data into $k$ folds. The test set stays unseen.

In [27]:
my_seeds = [0, 3, 5]  
num_folds = 5

pairs = list(zip(train_texts, train_labels))
my_cross_validation_sets = [
    experiment.cross_validation_split(pairs, k=num_folds, seed=seed)
    for seed in my_seeds
]

## 5. Architectures

- **Log-linear** softmax regression (`'logistic'`)
- **MLP** with one hidden layer of 128 units (`'mlp'`), trained for 10 epochs (as in the tutorial)
- **Small MLP** (`'small-mlp'`): the same MLP, but with only **16 units**. Same training settings. It tests whether a much smaller network is enough for this task.

Training keeps the epoch with the lowest validation loss.

In [ ]:
architectures = OrderedDict([
    ('logistic', dict(model_option='logistic', loglin_args={'max_iter': 100})),
    ('mlp', dict(model_option='mlp', smlp_args={
        'hidden_dim': 128, 'dropout': 0.1, 'batch_size': 64,
        'learning_rate': 1e-4, 'weight_decay': 1e-4, 'num_epochs': 10})),
    ('small-mlp', dict(model_option='mlp', smlp_args={
        'hidden_dim': 16, 'dropout': 0.1, 'batch_size': 64,
        'learning_rate': 1e-4, 'weight_decay': 1e-4, 'num_epochs': 10})),
])

## 6. Experiments

Train and evaluate every combination of representation and architecture with cross-validation.

In [29]:
%%time
my_classifiers = OrderedDict()

for arch_name, arch_args in architectures.items():
    for enc_name, encoder in encoders.items():
        sys_name = f"{arch_name}_{enc_name}"
        print(f"=== {sys_name} ===")
        my_classifiers[sys_name] = []
        for run, seed in enumerate(my_seeds):
            np.random.seed(seed)
            torch.manual_seed(seed)
            result = experiment.cross_validation_experiment(
                data_splits=my_cross_validation_sets[run],
                outcome_space=outcome_space,
                encoder=encoder,
                save_to=f"{models_folder}/{sys_name}/seed-{seed}",
                **arch_args,
            )
            my_classifiers[sys_name].append(result)

=== logistic_indicator ===


  0%|          | 0/5 [00:00<?, ?it/s]

/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/emmanouilsokorelis/Projects/uva-msai/nlp1/nlp1-student/.venv/lib/

  0%|          | 0/5 [00:00<?, ?it/s]

... (228 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6235 | train acc: 0.6909 | val loss: 0.5362 | val acc: 0.8014
Epoch 02 | train loss: 0.4737 | train acc: 0.8274 | val loss: 0.4287 | val acc: 0.8563
Epoch 03 | train loss: 0.3872 | train acc: 0.8679 | val loss: 0.3748 | val acc: 0.8665
Epoch 04 | train loss: 0.3344 | train acc: 0.8835 | val loss: 0.3421 | val acc: 0.8735
Epoch 05 | train loss: 0.3018 | train acc: 0.8938 | val loss: 0.3244 | val acc: 0.8763
Epoch 06 | train loss: 0.2784 | train acc: 0.9007 | val loss: 0.3135 | val acc: 0.8795
Epoch 07 | train loss: 0.2604 | train acc: 0.9083 | val loss: 0.3070 | val acc: 0.8813
Epoch 08 | train loss: 0.2447 | train acc: 0.9123 | val loss: 0.3027 | val acc: 0.8820
Epoch 09 | train loss: 0.2322 | train acc: 0.9146 | val loss: 0.3007 | val acc: 0.8830
Epoch 10 | train loss: 0.2212 | train acc: 0.9220 | val loss: 0.2998 | val acc: 0.8822
Training done! Time taken: 10.360 seconds
Transforming documents...
Document transformation done! Time taken: 4.034 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6175 | train acc: 0.6921 | val loss: 0.5366 | val acc: 0.8002
Epoch 02 | train loss: 0.4757 | train acc: 0.8312 | val loss: 0.4343 | val acc: 0.8508
Epoch 03 | train loss: 0.3909 | train acc: 0.8644 | val loss: 0.3764 | val acc: 0.8653
Epoch 04 | train loss: 0.3393 | train acc: 0.8820 | val loss: 0.3428 | val acc: 0.8759
Epoch 05 | train loss: 0.3066 | train acc: 0.8926 | val loss: 0.3234 | val acc: 0.8799
Epoch 06 | train loss: 0.2807 | train acc: 0.8995 | val loss: 0.3110 | val acc: 0.8837
Epoch 07 | train loss: 0.2629 | train acc: 0.9056 | val loss: 0.3029 | val acc: 0.8833
Epoch 08 | train loss: 0.2481 | train acc: 0.9120 | val loss: 0.2980 | val acc: 0.8821
Epoch 09 | train loss: 0.2339 | train acc: 0.9161 | val loss: 0.2954 | val acc: 0.8845
Epoch 10 | train loss: 0.2242 | train acc: 0.9193 | val loss: 0.2942 | val acc: 0.8849
Training done! Time taken: 10.251 seconds
CPU times: user 28min 27s, sys: 1min 11s, total: 29min 38s
Wall time: 28min 25s


Save the validation accuracies, so they are not lost if the kernel restarts:

In [30]:
cv_results_file = results_folder / "cv_accuracies.json"
with open(cv_results_file, "w") as f:
    json.dump({name: [{str(fold): float(acc) for fold, acc in run['accuracies'].items()} for run in runs]
               for name, runs in my_classifiers.items()}, f, indent=2)

## 7. Training history

`results/training_history.json` stores, for every system / seed / fold:

- **MLPs:** training and validation loss and accuracy for every epoch, and `best_epoch` (the epoch kept, lowest validation loss). These are the values printed in the training log above.
- **Logistic regression:** final training and validation loss and accuracy, and the number of solver iterations (it is fitted by a solver, not in epochs). These were computed from the saved models on the same folds.

Each fold folder in `results/models/` also has its own `history.json`. Load it like this:

In [31]:
with open(results_folder / "training_history.json") as f:
    training_history = json.load(f)

h = training_history["small-mlp_logcounts"]["seed-0"]["fold-0"]
print("best epoch:", h["best_epoch"])
for e in h["epochs"]:
    print(f"epoch {e['epoch']:2d} | train loss {e['train_loss']:.4f} | val loss {e['val_loss']:.4f} | val acc {e['val_acc']:.4f}")

best epoch: 10
epoch  1 | train loss 0.6518 | val loss 0.5898 | val acc 0.7928
epoch  2 | train loss 0.5194 | val loss 0.4626 | val acc 0.8452
epoch  3 | train loss 0.4154 | val loss 0.3919 | val acc 0.8641
epoch  4 | train loss 0.3566 | val loss 0.3539 | val acc 0.8745
epoch  5 | train loss 0.3185 | val loss 0.3302 | val acc 0.8761
epoch  6 | train loss 0.2930 | val loss 0.3165 | val acc 0.8795
epoch  7 | train loss 0.2751 | val loss 0.3085 | val acc 0.8799
epoch  8 | train loss 0.2590 | val loss 0.3016 | val acc 0.8832
epoch  9 | train loss 0.2453 | val loss 0.2979 | val acc 0.8830
epoch 10 | train loss 0.2338 | val loss 0.2965 | val acc 0.8840
